# Download Dataset


In [1]:
!pip install aicrowd-cli

In [2]:
API_KEY = 'cc0a3da7611cfc6098a7bd9db11b3ecf' # Please get your your API Key from [https://www.aicrowd.com/participants/me]
!aicrowd login --api-key $API_KEY

API Key valid
Saved API Key successfully!


In [3]:
# Downloading the Dataset
!mkdir data
!aicrowd dataset download --challenge emotion-detection -j 3 -o data

mkdir: cannot create directory ‘data’: File exists
train.csv:   0% 0.00/2.30M [00:00<?, ?B/s]
test.csv:   0% 0.00/642k [00:00<?, ?B/s]

val.csv:   0% 0.00/262k [00:00<?, ?B/s]

val.csv: 100% 262k/262k [00:00<00:00, 1.09MB/s]

test.csv: 100% 642k/642k [00:00<00:00, 2.00MB/s]
train.csv: 100% 2.30M/2.30M [00:00<00:00, 4.75MB/s]


# Download & Import Libraries

In [4]:
!pip install emoji

In [5]:
import os
import re
import emoji
import time
import numpy as np
import pandas as pd
import nltk
import torch
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from torch import nn, optim, FloatTensor
from torch.utils.data import Dataset, DataLoader

In [6]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Read Dataset

In [7]:
train_dataset = pd.read_csv("data/train.csv")
validation_dataset = pd.read_csv("data/val.csv")[1:]
test_dataset = pd.read_csv("data/test.csv")

train_dataset.head(20)

,text,label
0,takes no time to copy/paste a press release,0
1,You're delusional,1
2,Jazz fan here. I completely feel. Lindsay Mann...,0
3,ah i was also confused but i think they mean f...,0
4,Thank you so much. ♥️ that means a lot.,0
5,And I’ll be there!!!,0
6,There are some amazingly cringey compilations ...,0
7,Check the frame (FPS) limit option in the adva...,0
8,you made me think I was in the dbd subreddit w...,0
9,It was in your op.,0


# Process text

In [8]:
stops = set(stopwords.words('english'))
porter = nltk.PorterStemmer()

def pre_process(str):
    def rm_html_tags(str):
        html_prog = re.compile(r'<[^>]+>',re.S)
        return html_prog.sub('', str)

    def rm_html_escape_characters(str):
        pattern_str = r'&quot;|&amp;|&lt;|&gt;|&nbsp;|&#34;|&#38;|&#60;|&#62;|&#160;|&#20284;|&#30524;|&#26684|&#43;|&#20540|&#23612;'
        escape_characters_prog = re.compile(pattern_str, re.S)
        return escape_characters_prog.sub('', str)

    def rm_at_user(str):
        return re.sub(r'@[a-zA-Z_0-9]*', '', str)

    def rm_url(str):
        return re.sub(r'http[s]?:[/+]?[a-zA-Z0-9_\.\/]*', '', str)

    def rm_repeat_chars(str):
        return re.sub(r'(.)(\1){2,}', r'\1\1', str)

    def rm_hashtag_symbol(str):
        return re.sub(r'#', '', str)

    def rm_time(str):
        return re.sub(r'[0-9][0-9]:[0-9][0-9]', '', str)

    def rm_punctuation(str):
        return re.sub(r'[^\w\s]', ' ', str)

    def split_emojis(str):
        text_part = ''.join(c for c in str if c not in emoji.UNICODE_EMOJI)
        emoji_part = ' '.join(c for c in str if c in emoji.UNICODE_EMOJI)
        return text_part + ' ' + emoji_part
    
    # do not change the preprocessing order only if you know what you're doing 
    str = str.lower()
    str = rm_url(str)        
    str = rm_at_user(str)        
    str = rm_repeat_chars(str) 
    str = rm_hashtag_symbol(str)       
    str = rm_time(str)
    str = rm_punctuation(str)
    # str = emoji.demojize(str, delimiters=(' emoji_', ' '))
    str = split_emojis(str)

    try:
        str = nltk.tokenize.word_tokenize(str)
        try:
            str = [porter.stem(t) for t in str]
        except:
            pass
    except:
        pass

    words = [w for w in str if w and w not in stops]
    return ' '.join(words)

In [9]:
text = "takes no time to copy/paste a press release"
pre_process(text)

'take time copi past press releas'

In [10]:
start_time = time.time()
train_dataset['processed_text'] = train_dataset['text'].apply(
    lambda x: pre_process(x)
)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

train_dataset.head(20)

Elapsed time: 12.6896 seconds


,text,label,processed_text
0,takes no time to copy/paste a press release,0,take time copi past press releas
1,You're delusional,1,delusion
2,Jazz fan here. I completely feel. Lindsay Mann...,0,jazz fan complet feel lindsay mann cousin ha v...
3,ah i was also confused but i think they mean f...,0,ah wa also confus think mean friend around age
4,Thank you so much. ♥️ that means a lot.,0,thank much mean lot
5,And I’ll be there!!!,0,
6,There are some amazingly cringey compilations ...,0,amazingli cringey compil terribl dialogu thi s...
7,Check the frame (FPS) limit option in the adva...,0,check frame fp limit option advanc graphic opt...
8,you made me think I was in the dbd subreddit w...,0,made think wa dbd subreddit statement idk whi
9,It was in your op.,0,wa op


In [11]:
start_time = time.time()
validation_dataset['processed_text'] = validation_dataset['text'].apply(
    lambda x: pre_process(x)
)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

validation_dataset.head(20)

Elapsed time: 1.4322 seconds


,text,label,processed_text
1,im still starving,1,im still starv
2,*Hey just noticed..* it's your **2nd Cakeday**...,0,hey notic 2nd cakeday slumbishop hug
3,They just did. Check out the sticky post.,0,check sticki post
4,"I hope so too, she deserves it.",0,hope deserv
5,is it dangerous to take a quick photo while st...,0,danger take quick photo stop red light
6,I’m in my second year. Still closeted. Still u...,1,second year still closet still uncomfort bodi ...
7,"Noted, I've been looking into that",0,note look
8,Thank you for saying that I just haven’t felt ...,0,thank say felt right sad constantli
9,"Screw the watch stuff, I wanna hear about the ...",1,screw watch stuff wan na hear porn
10,"Exactly....we need a car tunnel, and then let ...",0,exactli need car tunnel let old road allow tru...


# Create TF-IDF Features

In [12]:
print(train_dataset.shape)
print(validation_dataset.shape)
combined_dataset = pd.concat([train_dataset, validation_dataset])
print(combined_dataset.shape)

(31255, 3)
(3472, 3)
(34727, 3)


In [13]:
tfidf_vect = TfidfVectorizer(analyzer='word', token_pattern=r'\w{1,}', max_features=5000)
tfidf_vect.fit(combined_dataset['processed_text'])

TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.float64'>, encoding='utf-8',
                input='content', lowercase=True, max_df=1.0, max_features=5000,
                min_df=1, ngram_range=(1, 1), norm='l2', preprocessor=None,
                smooth_idf=True, stop_words=None, strip_accents=None,
                sublinear_tf=False, token_pattern='\\w{1,}', tokenizer=None,
                use_idf=True, vocabulary=None)

In [14]:
start_time = time.time()
xtrain_tfidf =  tfidf_vect.transform(train_dataset['processed_text'])
xval_tfidf =  tfidf_vect.transform(validation_dataset['processed_text'])
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

print(xtrain_tfidf.shape)
print(xval_tfidf.shape)

Elapsed time: 0.3095 seconds
(31255, 5000)
(3472, 5000)


# Transform Data

In [15]:
class trainData(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data
        
    def __getitem__(self, index):
        return self.X_data[index], self.y_data[index]
        
    def __len__ (self):
        return len(self.X_data)

## test data    
class testData(Dataset):
    def __init__(self, X_data):
        self.X_data = X_data
        
    def __getitem__(self, index):
        return self.X_data[index]
        
    def __len__ (self):
        return len(self.X_data)

In [16]:
y_train = train_dataset['label'].values
y_val = validation_dataset['label'].values

train_data = trainData(
    FloatTensor(xtrain_tfidf.toarray()), 
    FloatTensor(y_train)
)
val_data = trainData(
    FloatTensor(xval_tfidf.toarray()), 
    FloatTensor(y_val)
)

In [31]:
## Initialize dataloader
LEARNING_RATE = 0.001
BATCH_SIZE = 512
NUM_EPOCHS = 100

train_loader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=1)

# Train Model

In [18]:
train_dataset['label'].value_counts()

0    24718
1     6537
Name: label, dtype: int64

In [27]:
del model

In [28]:
# Hyperparameters for our network
input_size = xtrain_tfidf.shape[1]
hidden_sizes = [128, 64]
output_size = 1

# Build a feed-forward network
model = nn.Sequential(
    nn.Linear(input_size, hidden_sizes[0]),
    nn.ReLU(),
    nn.Linear(hidden_sizes[0], hidden_sizes[1]),
    nn.ReLU(),
    nn.Linear(hidden_sizes[1], output_size),
    nn.Sigmoid()
)
print(model)

Sequential(
  (0): Linear(in_features=5000, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=1, bias=True)
  (5): Sigmoid()
)


In [29]:
criterion = nn.BCEWithLogitsLoss(pos_weight = torch.FloatTensor ([4.0]))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [32]:
def eval(y_pred, y_true):
    y_pred_tag = torch.round(torch.sigmoid(y_pred))
    tp = (y_true * y_pred).sum().to(torch.float32)
    tn = ((1 - y_true) * (1 - y_pred)).sum().to(torch.float32)
    fp = ((1 - y_true) * y_pred).sum().to(torch.float32)
    fn = (y_true * (1 - y_pred)).sum().to(torch.float32)
    
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f1 = 2 * (precision*recall) / (precision + recall + epsilon)
    acc = (tp + tn) / (tp + tn + fp + fn)
    
    return f1, acc

In [33]:
N_EPOCHS_STOP = 6
min_val_loss = np.Inf
epochs_no_improve = 0

for e in range(1, NUM_EPOCHS+1):
    train_loss = val_loss = 0
    train_acc = val_acc = 0
    train_f1 = val_f1 = 0
    for X_batch, y_batch in train_loader:
        # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()        
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch.unsqueeze(1))
        f1, acc = eval(y_pred, y_batch.unsqueeze(1))
        
        loss.backward() # Calculate gradients
        optimizer.step() # Update parameters
        
        train_loss += loss.item()
        train_acc += acc
        train_f1 += f1

    for X_batch, y_batch in val_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch.unsqueeze(1))
        f1, acc = eval(y_pred, y_batch.unsqueeze(1))
        val_loss += loss.item()
        val_acc += acc
        val_f1 += f1

    print(
        f'Epoch {e+0:03}: | \
        Training Loss: {train_loss/len(train_loader):.3f} | \
        F1 Score: {train_f1/len(train_loader):.3f} | \
        Acc: {train_acc/len(train_loader):.3f}  | \
        Validation Loss: {val_loss/len(val_loader):.3f} | \
        F1 Score: {val_f1/len(val_loader):.3f} | \
        Acc: {val_acc/len(val_loader):.3f}'
    )

    # Early stopping
    if val_loss/len(val_loader) < min_val_loss:
        # torch.save(model)  # Save the model
        epochs_no_improve = 0
        min_val_loss = val_loss/len(val_loader)
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= N_EPOCHS_STOP:
        break

Epoch 001: |         Training Loss: 1.147 |         F1 Score: 0.281 |         Acc: 0.569  |         Validation Loss: 1.104 |         F1 Score: 0.074 |         Acc: 0.717
Epoch 002: |         Training Loss: 1.058 |         F1 Score: 0.376 |         Acc: 0.746  |         Validation Loss: 1.039 |         F1 Score: 0.114 |         Acc: 0.765
Epoch 003: |         Training Loss: 0.980 |         F1 Score: 0.572 |         Acc: 0.808  |         Validation Loss: 1.017 |         F1 Score: 0.123 |         Acc: 0.788
Epoch 004: |         Training Loss: 0.953 |         F1 Score: 0.657 |         Acc: 0.843  |         Validation Loss: 1.017 |         F1 Score: 0.127 |         Acc: 0.783
Epoch 005: |         Training Loss: 0.935 |         F1 Score: 0.690 |         Acc: 0.860  |         Validation Loss: 1.018 |         F1 Score: 0.124 |         Acc: 0.786
Epoch 006: |         Training Loss: 0.926 |         F1 Score: 0.712 |         Acc: 0.870  |         Validation Loss: 1.021 |         F1 Score: 0.117 |

# Submit Results

In [36]:
test_dataset['processed_text'] = test_dataset['text'].apply(
    lambda x: pre_process(x)
)
xtest_tfidf =  tfidf_vect.transform(test_dataset['processed_text'])
test_data = testData(
    FloatTensor(xtest_tfidf.toarray())
)
print(xtest_tfidf.shape)

test_loader = DataLoader(dataset=test_data, batch_size=1)

(8682, 5000)


In [38]:
y_pred_list = []
model.eval()
with torch.no_grad():
    for X_batch in test_loader:
        # X_batch = X_batch.to(device)
        y_test_pred = model(X_batch)
        y_test_pred = torch.sigmoid(y_test_pred)
        y_pred_tag = torch.round(y_test_pred)
        y_pred_list.append(y_pred_tag.cpu().numpy())

y_pred_list = [a.squeeze().tolist() for a in y_pred_list]
test_dataset['label'] = y_pred_list
test_dataset.head()

,text,label,processed_text
0,I was already over the edge with Cassie Zamora...,1.0,wa alreadi edg cassi zamora show disdain two t...
1,I think you're right. She has oodles of cash a...,0.0,think right ha oodl cash young grandchildren e...
2,Haha I love this. I used to give mine phone bo...,0.0,haha love thi use give mine phone book room wo...
3,Probably out of desperation as they going no a...,1.0,probabl desper go answer made god
4,Sorry !! You’re real good at that!!,1.0,sorri real good


In [39]:
!mkdir assets

# Saving the sample submission in assets directory
if 'processed_text' in test_dataset.columns:
    test_dataset.drop(columns=['processed_text'], inplace=True)

test_dataset.to_csv(os.path.join("assets", "submission.csv"), index=False)

In [40]:
!aicrowd notebook submit -c emotion-detection -a assets --no-verify

Mounting Google Drive 💾
Your Google Drive will be mounted to access the colab notebook
Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.activity.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fexperimentsandconfigs%20https%3a%2f%2fwww.googleapis.com%2fauth%2fphotos.native&response_type=code

Enter your authorization code:
4/1AY0e-g7sPRpMdKfeD1eDVKEuHW3oS1LsJ8j65llujwQiCtkkCP05i3fdEzA
Mounted at /content/drive
Using notebook: /content/drive/MyDrive/Colab Notebooks/tfidf-ann-classifier.ipynb for submission...
Scrubbing API keys from the noteb

# References

#### Tutorial - Early Stopping - Vanilla RNN - PyTorch
https://www.kaggle.com/akhileshrai/tutorial-early-stopping-vanilla-rnn-pytorch

#### PyTorch [Tabular] — Binary Classification
https://towardsdatascience.com/pytorch-tabular-binary-classification-a0368da5bb89